> **Superseded.** The scores in this notebook were later found to be inflated. Some segments are near-copies of each other (same recording, same time) and the folds did not keep them together (see notebook 06). With near-duplicate segments kept together, the forest's AUC fell from about 0.78 to about 0.5 in a first check. The findings below describe what the model used under the old folds, and are likely explained by matching near-duplicate waveforms. Kept as a record of the investigation.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter1d
from sklearn.ensemble import RandomForestClassifier

from src.config import PROCESSED, RANDOM_STATE, SAMPLING_RATE_HZ
from src.cv import load_fold_table
from src.data import feature_cols
from src.evaluate import cross_validate

dev = pd.read_csv(PROCESSED / "dev_2v3.csv")
cols = feature_cols(dev)
X = dev[cols].to_numpy(dtype=float)
table = load_fold_table()
make_forest = lambda: RandomForestClassifier(50, random_state=RANDOM_STATE, n_jobs=-1)

In [2]:
rng = np.random.default_rng(0)
Xc = X - X.mean(axis=1, keepdims=True)          # each chunk with its average level removed
Z = Xc / Xc.std(axis=1, keepdims=True)          # ... and rescaled to equal spread
smooth = uniform_filter1d(X, size=9, axis=1, mode="nearest")      # 9-sample moving average
power = np.abs(np.fft.rfft(Xc, axis=1))[:, 1:] ** 2               # power at 89 frequencies

def acf(A, nlag=20):
    """How similar each chunk is to itself shifted by 1, 2, ... nlag samples."""
    n = A.shape[1]
    return np.stack([(A[:, :n - l] * A[:, l:]).mean(axis=1) for l in range(1, nlag + 1)], axis=1)

variants = {
    "raw (reference)": X,
    "time order shuffled inside each chunk": np.apply_along_axis(rng.permutation, 1, X),
    "chunk mean removed": Xc,
    "chunk mean removed, equal spread": Z,
    "first differences (slopes)": np.diff(X, axis=1),
    "smoothed only (slow part)": smooth,
    "raw minus smoothed (fast part)": X - smooth,
    "autocorrelation, lags 1-20": acf(Z),
    "log spectrum (89 numbers)": np.log(power + 1e-9),
    "relative log spectrum (shape only)": np.log(power / power.sum(axis=1, keepdims=True) + 1e-9),
    "chunk number only (sanity)": dev["chunk"].to_numpy(dtype=float).reshape(-1, 1),
}

In [3]:
rows = []
for name, F in variants.items():
    r = cross_validate(make_forest, dev, F, table)       # 5 repeats, same folds for every version
    rows.append({"version of the chunk": name, "AUC": r["auc"].mean(), "AUC sd": r["auc"].std(),
                 "accuracy": r["accuracy"].mean(), "chunk accuracy": r["chunk_accuracy"].mean()})
pd.DataFrame(rows).round(3)

,version of the chunk,AUC,AUC sd,accuracy,chunk accuracy
0,raw (reference),0.778,0.021,0.690,0.608
1,time order shuffled inside each chunk,0.518,0.015,0.512,0.513
2,chunk mean removed,0.883,0.014,0.774,0.673
3,"chunk mean removed, equal spread",0.886,0.010,0.782,0.660
4,first differences (slopes),0.789,0.019,0.702,0.604
5,smoothed only (slow part),0.779,0.018,0.698,0.601
6,raw minus smoothed (fast part),0.772,0.014,0.706,0.600
7,"autocorrelation, lags 1-20",0.724,0.008,0.672,0.561
8,log spectrum (89 numbers),0.776,0.011,0.706,0.613
9,relative log spectrum (shape only),0.779,0.018,0.700,0.598


In [4]:
F = np.log(power / power.sum(axis=1, keepdims=True) + 1e-9)             # shape-only spectrum, 89 numbers
freq = np.arange(1, F.shape[1] + 1) * SAMPLING_RATE_HZ / X.shape[1]     # frequency of each number, in Hz
bands = {"delta 0.5-4": (0.5, 4), "theta 4-8": (4, 8), "alpha 8-13": (8, 13),
         "beta 13-30": (13, 30), "gamma 30-40": (30, 40), "above 40": (40, 99)}

rows = [{"band": "all", "AUC if dropped": cross_validate(make_forest, dev, F, table)["auc"].mean()}]
for name, (lo, hi) in bands.items():
    inside = (freq >= lo) & (freq < hi)
    rows.append({"band": name,
                 "AUC if dropped": cross_validate(make_forest, dev, F[:, ~inside], table)["auc"].mean(),
                 "AUC if kept alone": cross_validate(make_forest, dev, F[:, inside], table)["auc"].mean()})
pd.DataFrame(rows).round(3)

,band,AUC if dropped,AUC if kept alone
0,all,0.779,NaN
1,delta 0.5-4,0.759,0.599
2,theta 4-8,0.754,0.657
3,alpha 8-13,0.741,0.719
4,beta 13-30,0.762,0.692
5,gamma 30-40,0.799,0.605
6,above 40,0.796,0.680


# 05 Ablation: what is the forest using?

Same forest (50 trees), same folds, 5 repeats. Scores are segment-level AUC and accuracy, averaged over repeats.

## Versions of the chunk
| Version | AUC | Accuracy |
|---|---|---|
| Raw (reference) | 0.778 | 0.690 |
| Time order shuffled inside each chunk | 0.518 | 0.512 |
| Chunk mean removed | 0.883 | 0.774 |
| Chunk mean removed, equal spread | 0.886 | 0.782 |
| First differences | 0.789 | 0.702 |
| Smoothed only (slow part) | 0.779 | 0.698 |
| Raw minus smoothed (fast part) | 0.772 | 0.706 |
| Autocorrelation, lags 1-20 | 0.724 | 0.672 |
| Log spectrum (89 numbers) | 0.776 | 0.706 |
| Relative log spectrum (shape only) | 0.779 | 0.700 |
| Chunk number only (sanity) | 0.500 | 0.500 |

## Frequency bands (relative spectrum)
| Band (Hz) | AUC if dropped | AUC if kept alone |
|---|---|---|
| All | 0.779 | |
| Delta 0.5-4 | 0.759 | 0.599 |
| Theta 4-8 | 0.754 | 0.657 |
| Alpha 8-13 | 0.741 | 0.719 |
| Beta 13-30 | 0.762 | 0.692 |
| Gamma 30-40 | 0.799 | 0.605 |
| Above 40 | 0.796 | 0.680 |

## Findings
- Shuffling the order of samples inside a chunk drops the AUC to chance (0.52). The forest uses the waveform's shape, not the size or spread of the values.
- Removing each chunk's average level and spread raises the AUC to 0.89. Size and offset get in the way.
- Slow part only, fast part only and slopes all keep 0.77 to 0.79. No single speed of change carries the information.
- Spectrum numbers recover the full score (0.78). Twenty autocorrelation numbers recover most of it (0.72).
- No single band is essential (dropping any one costs at most 0.04), and every band alone keeps part of the score (0.60 to 0.72). The information is spread across frequencies. Alpha is the strongest band alone (0.72).
- Dropping gamma or the bands above 40 Hz does not hurt (0.80 vs 0.78). Differences of about 0.02 are within the spread between repeats.
- Above 40 Hz alone scores 0.68, although that range holds about 0.16% of the power. It could be recording noise or interference. Not tested.
- Chunk number alone gives 0.50, so there is no leak through chunk order.

## Caveats
- About 11 versions plus 12 band runs were tried, so the best scores (0.89) are optimistic. Differences below about 0.02 are noise.
- This shows what the forest uses (shape), not why it works. A real difference between the two regions cannot be separated from a recording signature with this data.